In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [ ]:
"""
H4 Analysis: Cross-Language License Frictions
------------------------------------------------------------------------
H4. Method-level license compatibility frictions are unevenly
distributed across programming languages.

Design notes:
- Operates at the relationship level on language and LCD category, so no project-identity resolution is required.
- Friction is Categories 3 and 4 (actionable debt) -- the narrower definition, matching H2. H5 uses a broader one that also counts
  Category 5; see the paper's Study Design section.
- Category 0 is NOT excluded here: it counts as non-friction, since a relationship with no provenance match has no license conflict.
  H5 excludes it instead, which is why the two hypotheses report different N.
- N is derived from the pipeline rather than hardcoded, so the deduplicated total is reported automatically.
- Because relationships cluster within repositories, the omnibus association is re-checked at the repository level; the per-language
  odds ratios are not robust to that adjustment and are reported descriptively.

Reproduces: N = 1,116,445; chi2(5) = 5,939.21, p < .001, Cramer's V = 0.0729. Friction rates 8.19% (Java) to 17.13% (C#); odds ratios
0.524 (Java) and 1.213 (C#) against the C baseline. Repository level: 5,362 repositories, H = 113.25, epsilon^2 = 0.020.
"""

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf

FILE_PATH = DATA / "license_analysis_results_processed.csv"
TARGET_LANGS = ["C", "C++", "C#", "Java", "JavaScript", "Python"]
DEDUP_KEYS = ["method_hash", "source_repository_url", "sink_repository_url"]


def cramers_v_corrected(contingency_table):
    """Bias-corrected Cramer's V -- consistent with the methodology
    used for H2/H5, for consistency across the paper."""
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    r, c = contingency_table.shape
    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - ((c - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    v = np.sqrt(phi2_corr / min(c_corr - 1, r_corr - 1))
    return v, chi2, p, dof


def evaluate_h4():
    print("==================================================")
    print("  H4 STATISTICAL EVALUATION (LCD CATEGORIES 3 & 4)")
    print("==================================================")

    df = pd.read_csv(FILE_PATH)
    df = df.replace(r"^\s*$", np.nan, regex=True)

    before = len(df)
    df = df.drop_duplicates(subset=DEDUP_KEYS)
    print(f"[dedup] {before} -> {len(df)} rows ({before - len(df)} duplicates removed)")

    df["language"] = df["language"].replace({"JS": "JavaScript"})
    df = df[df["language"].isin(TARGET_LANGS)].copy()

    df["violation_lcd_category"] = pd.to_numeric(df["violation_lcd_category"], errors="coerce")

    # Category 3: Incompatible-license reuse; Category 4: High-risk conflict
    df["is_friction"] = df["violation_lcd_category"].isin([3, 4]).astype(int)

    df_clean = df.dropna(subset=["language"]).copy()

    print(f"\nTotal Method Observations Analyzed: {len(df_clean):,}")
    print(f"Target Languages ({len(TARGET_LANGS)}): {TARGET_LANGS}\n")

    # ------------------------------------------------------------------
    # [1] Descriptive Summary & Rates by Language
    # ------------------------------------------------------------------
    lcd_counts = pd.crosstab(df_clean["language"], df_clean["violation_lcd_category"])
    print("[1] LCD Category Distribution by Language (Counts):")
    print(lcd_counts.to_string())
    print("\n" + "-" * 50)

    per_lang_totals = df_clean.groupby("language").size()
    friction_rates = (
        pd.crosstab(df_clean["language"], df_clean["is_friction"], normalize="index")[1] * 100
    )

    print("\n[2] Actionable Friction Rate (Cat 3 + 4) by Language:")
    summary_rows = []
    for lang in TARGET_LANGS:
        rate = friction_rates.get(lang, np.nan)
        total = per_lang_totals.get(lang, 0)
        print(f"  - {lang:<15}: {rate:.2f}% actionable friction (N={total:,})")
        summary_rows.append({"language": lang, "total_methods": total, "friction_rate_pct": rate})

    # ------------------------------------------------------------------
    # [3] Chi-Square Test of Independence + Cramer's V
    # ------------------------------------------------------------------
    contingency = pd.crosstab(df_clean["language"], df_clean["is_friction"])
    v, chi2, p_val, dof = cramers_v_corrected(contingency)

    print("\n" + "=" * 50)
    print("CHI-SQUARE TEST RESULTS (H4)")
    print("=" * 50)
    print(f"N (total, matches Table V row sum): {len(df_clean):,}")
    print(f"Chi-Square Statistic (chi2): {chi2:.4f}")
    print(f"Degrees of Freedom (dof): {dof}")
    print(f"p-value: {p_val:.4e}")
    print(f"Cramer's V (bias-corrected): {v:.4f}")

    # ------------------------------------------------------------------
    # [4] Logistic Regression: Odds Ratios by Language (C as baseline)
    # ------------------------------------------------------------------
    print("\n" + "=" * 50)
    print("LOGISTIC REGRESSION: FRICTION ODDS BY LANGUAGE (baseline = C)")
    print("=" * 50)

    df_clean["lang_factor"] = pd.Categorical(
        df_clean["language"], categories=TARGET_LANGS, ordered=False
    )

    model = smf.logit("is_friction ~ C(lang_factor, Treatment(reference='C'))", data=df_clean).fit(disp=False)

    results_df = pd.DataFrame({
        "Odds Ratio": np.exp(model.params),
        "2.5% CI": np.exp(model.conf_int()[0]),
        "97.5% CI": np.exp(model.conf_int()[1]),
        "p-value": model.pvalues,
    })
    print(results_df.round(4).to_string())

    # ------------------------------------------------------------------
    # [5] Summary table matching manuscript Table V structure
    # ------------------------------------------------------------------
    summary_df = pd.DataFrame(summary_rows).set_index("language").reindex(TARGET_LANGS)
    print("\n[5] Summary table (for cross-check against manuscript Table V):")
    print(summary_df.to_string())
    print(f"\nColumn sum of total_methods (should equal N above): "
          f"{summary_df['total_methods'].sum():,}")

    return {
        "N": len(df_clean),
        "contingency": contingency,
        "chi2": chi2,
        "p_value": p_val,
        "cramers_v": v,
        "logit_model": model,
        "logit_results": results_df,
        "summary_table": summary_df,
    }


if __name__ == "__main__":
    results = evaluate_h4()

    results["summary_table"].to_csv(RESULTS / "h4_language_summary.csv")
    results["logit_results"].to_csv(RESULTS / "h4_logit_odds_ratios.csv")
    results["contingency"].to_csv(RESULTS / "h4_contingency_table.csv")

    print("\n[done] Results written to h4_language_summary.csv, "
          "h4_logit_odds_ratios.csv, h4_contingency_table.csv")

  H4 STATISTICAL EVALUATION (LCD CATEGORIES 3 & 4)
[dedup] 1183182 -> 1183182 rows (0 duplicates removed)

Total Method Observations Analyzed: 1,116,445
Target Languages (6): ['C', 'C++', 'C#', 'Java', 'JavaScript', 'Python']

[1] LCD Category Distribution by Language (Counts):
violation_lcd_category       0      1      2      3     4       5
language                                                         
C                       106994  54830  28754  52466  9113  170778
C#                       10623  11522   3626   7164   754   12521
C++                      70969  23004   8281  29472  1759  137851
Java                     45738  38761   4297  11615  2116   65033
JavaScript                8133   4208   1891   2592   306   12786
Python                   47418  32683  12284  18100  3099   64904

--------------------------------------------------

[2] Actionable Friction Rate (Cat 3 + 4) by Language:
  - C              : 14.56% actionable friction (N=422,935)
  - C++            : 11.51

In [ ]:
"""
H4 Clustering Robustness: cluster-robust inference + repository-level
aggregation
--------------------------------------------------------------------------
WHY

H4's chi-square test and logistic regression treat each provenance
relationship as an independent observation. But relationships cluster within
repositories: a single repository contributes many relationships that share
its language, its maintainers, and its licensing practices. The same
objection that overturned H5's inferential claim applies here.

This mirrors the H5 robustness analysis exactly, so the two are comparable:

  1. CLUSTER-ROBUST LOGIT. All observations retained; sandwich standard
     errors clustered on repository (base_repository_url). Corrects the
     calibration of the per-language odds ratios.

  2. REPOSITORY-LEVEL AGGREGATION. One friction rate per repository, with
     the repository's dominant language as its label. The REPOSITORY becomes
     the unit of analysis, removing within-repository dependence entirely.
     Kruskal-Wallis across the six languages, with epsilon-squared as a
     distribution-free effect size.

-- H4 uses a narrower friction definition than H5:
    H4: friction = Categories 3 and 4          (actionable debt)
    H5: friction = Categories 3, 4, and 5      (includes latent debt)
and H4 does NOT exclude Category 0 (it counts as non-friction), unlike H5.
Both choices replicate the published H4 pipeline; the sanity check below
verifies the reproduction before any model is fitted.
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import kruskal

FILE_PATH = DATA / "license_analysis_results_processed.csv"
DEDUP_KEYS = ["method_hash", "source_repository_url", "sink_repository_url"]
TARGET_LANGS = ["C", "C++", "C#", "Java", "JavaScript", "Python"]
FRICTION_CATEGORIES = [3, 4]     # H4's definition -- narrower than H5's
BASELINE_LANG = "C"
MIN_RELATIONSHIPS_PER_REPO = 5   # repos below this give unstable per-repo rates

EXPECTED = {"C": 14.56, "C#": 17.13, "Python": 11.88,
            "C++": 11.51, "JavaScript": 9.69, "Java": 8.19}


def epsilon_squared(groups):
    """Distribution-free effect size for Kruskal-Wallis. 0 = no difference."""
    all_v = np.concatenate(groups)
    n = len(all_v)
    h, _ = kruskal(*groups)
    return (h - len(groups) + 1) / (n - len(groups)) if n > len(groups) else np.nan


def build(path=FILE_PATH):
    df = pd.read_csv(path)
    df = df.drop_duplicates(subset=DEDUP_KEYS)
    df["language"] = df["language"].replace({"JS": "JavaScript"})
    df = df[df["language"].isin(TARGET_LANGS)].copy()
    # NOTE: Category 0 is NOT excluded here -- this replicates H4's pipeline.
    df["is_friction"] = df["violation_lcd_category"].isin(FRICTION_CATEGORIES).astype(int)
    df = df.dropna(subset=["language", "base_repository_url"])

    rates = df.groupby("language")["is_friction"].mean().mul(100).round(2)
    print(f"N = {len(df):,} relationships   (expected 1,116,445)")
    print("\nFriction rate by language (%):")
    for lang in TARGET_LANGS:
        got, exp = rates.get(lang, float("nan")), EXPECTED[lang]
        mark = "ok" if abs(got - exp) < 0.05 else "MISMATCH"
        print(f"  {lang:<12} {got:>6.2f}   expected {exp:>6.2f}   [{mark}]")
    ok = all(abs(rates.get(l, -1) - EXPECTED[l]) < 0.05 for l in TARGET_LANGS)
    print("\n[ok] reproduces published H4\n" if ok
          else "\n[!] MISMATCH -- resolve before trusting anything below\n")
    return df


def run(path=FILE_PATH):
    df = build(path)
    n_repo = df["base_repository_url"].nunique()
    print(f"Repositories (clusters): {n_repo:,}")
    print(f"Mean relationships per repository: {len(df)/n_repo:.1f}\n")

    # ---- 1. Naive vs cluster-robust logit -------------------------------
    print("=" * 72)
    print("1. LOGIT: per-language odds ratios, naive vs cluster-robust")
    print("=" * 72)
    others = [l for l in TARGET_LANGS if l != BASELINE_LANG]
    X = pd.DataFrame({f"lang_{l}": (df["language"] == l).astype(int) for l in others})
    X = sm.add_constant(X)
    y = df["is_friction"]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        naive = sm.Logit(y, X).fit(disp=False)
        clust = sm.Logit(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": df["base_repository_url"].astype("category").cat.codes},
            disp=False)

    rows = []
    for l in others:
        k = f"lang_{l}"
        rows.append({
            "language": l,
            "OR": round(float(np.exp(naive.params[k])), 3),
            "SE naive": round(float(naive.bse[k]), 4),
            "SE clust": round(float(clust.bse[k]), 4),
            "inflation": round(float(clust.bse[k] / naive.bse[k]), 1),
            "p naive": f"{float(naive.pvalues[k]):.3g}",
            "p clust": f"{float(clust.pvalues[k]):.3g}",
            "sig@.05": bool(float(clust.pvalues[k]) < .05),
        })
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    n_sig = int(out["sig@.05"].sum())
    print(f"\n{n_sig}/{len(others)} languages remain significant vs {BASELINE_LANG} "
          f"after clustering on repository.")

    # ---- 2. Repository-level aggregation --------------------------------
    print("\n" + "=" * 72)
    print("2. REPOSITORY-LEVEL AGGREGATION (repository = unit of analysis)")
    print("=" * 72)
    per_repo = (df.groupby("base_repository_url")
                  .agg(friction_rate=("is_friction", "mean"),
                       n_rel=("is_friction", "size"),
                       language=("language", lambda s: s.mode().iloc[0]))
                  .reset_index())
    before = len(per_repo)
    per_repo = per_repo[per_repo["n_rel"] >= MIN_RELATIONSHIPS_PER_REPO]
    print(f"Repositories: {before:,} -> {len(per_repo):,} "
          f"(>= {MIN_RELATIONSHIPS_PER_REPO} relationships)\n")

    summary = (per_repo.groupby("language")["friction_rate"]
               .agg(n="size", median="median", mean="mean").round(4))
    print(summary.to_string())

    groups = [per_repo.loc[per_repo["language"] == l, "friction_rate"].values
              for l in TARGET_LANGS
              if (per_repo["language"] == l).sum() >= 5]
    if len(groups) >= 3:
        h, p = kruskal(*groups)
        eps = epsilon_squared(groups)
        mag = ("negligible" if eps < .01 else "small" if eps < .06
               else "medium" if eps < .14 else "large")
        print(f"\nKruskal-Wallis H = {h:.2f}, p = {p:.4g}")
        print(f"Epsilon-squared  = {eps:.4f} ({mag})")
        print("\n=> " + ("Language differences persist at the repository level."
                         if p < .05 else
                         "NOT significant at the repository level -- the language "
                         "effect may reflect within-repository clustering."))
    else:
        print("\n[!] Too few languages with enough repositories to test.")

    per_repo.to_csv(RESULTS / "h4_repo_level_rates.csv", index=False)
    print("\nSaved per-repository rates: h4_repo_level_rates.csv")
    return per_repo


if __name__ == "__main__":
    run()

N = 1,116,445 relationships   (expected 1,116,445)

Friction rate by language (%):
  C             14.56   expected  14.56   [ok]
  C++           11.51   expected  11.51   [ok]
  C#            17.13   expected  17.13   [ok]
  Java           8.19   expected   8.19   [ok]
  JavaScript     9.69   expected   9.69   [ok]
  Python        11.88   expected  11.88   [ok]

[ok] reproduces published H4

Repositories (clusters): 9,322
Mean relationships per repository: 119.8

1. LOGIT: per-language odds ratios, naive vs cluster-robust
  language    OR  SE naive  SE clust  inflation   p naive p clust  sig@.05
       C++ 0.763    0.0074    0.2673       36.0 1.81e-289   0.312    False
        C# 1.213    0.0131    0.3960       30.2  2.13e-49   0.625    False
      Java 0.524    0.0099    0.2907       29.3         0  0.0261     True
JavaScript 0.629    0.0200    0.2439       12.2 3.22e-118  0.0577    False
    Python 0.791    0.0085    0.2394       28.1 5.21e-167   0.327    False

1/5 languages remain